# Fase 2 — referências de população e clima
Verifica hashes, unicidade e igualdade integral antes de disponibilizar tabelas Delta.

In [ ]:
import hashlib, json, uuid
from pathlib import Path
from datetime import datetime, timezone
from pyspark.sql import functions as F

inputs = {'population': {'sha256': '9c55ad8a681afa39059d8f7d40ffe41bbfa26f7b3d35b802238e4d14e433ab08'}, 'weather_municipal_weekly': {'sha256': '4b712406c2d4ff275f766d0c12448245d0426a2b07d0291422e4539a6129da97'}}
files = Path('/lakehouse/default/Files')
started = datetime.now(timezone.utc).isoformat()
run_id = str(uuid.uuid4())
result = {'run_id': run_id, 'environment': 'fabric', 'started_at_utc': started, 'inputs': {}}
for name, spec in inputs.items():
    relative = 'context/' + spec['sha256'] + '/' + name + '.parquet'
    path = files / relative
    actual_hash = hashlib.sha256(path.read_bytes()).hexdigest()
    if actual_hash != spec['sha256']:
        raise ValueError('Arquivo de contexto difere do aprovado: ' + name)
    incoming = spark.read.parquet('Files/' + relative)
    keys = ['municipio_ibge', 'ano_referencia'] if name == 'population' else ['municipio_ibge', 'week_start']
    if incoming.groupBy(*keys).count().where('count > 1').limit(1).count():
        raise ValueError('Chave de contexto duplicada: ' + name)
    table = 'f2_context_' + name
    if spark.catalog.tableExists(table):
        existing = spark.table(table)
        if incoming.schema != existing.schema or incoming.exceptAll(existing).limit(1).count() or existing.exceptAll(incoming).limit(1).count():
            raise ValueError('Tabela existente difere do snapshot: ' + table)
        action = 'reused'
    else:
        incoming.write.format('delta').mode('errorifexists').saveAsTable(table)
        action = 'created'
    stored = spark.table(table)
    if incoming.exceptAll(stored).limit(1).count() or stored.exceptAll(incoming).limit(1).count():
        raise ValueError('Tabela publicada difere do arquivo: ' + table)
    result['inputs'][name] = {'sha256': actual_hash, 'rows': stored.count(), 'table': table, 'action': action}
result['status'] = 'context_reference_tables_validated'
result['finished_at_utc'] = datetime.now(timezone.utc).isoformat()
audit = files / 'audit' / 'phase2_context' / run_id
audit.mkdir(parents=True, exist_ok=False)
(audit / 'summary.json').write_text(json.dumps(result, ensure_ascii=False, indent=2))
print(json.dumps(result, ensure_ascii=False))
mssparkutils.notebook.exit(json.dumps(result, ensure_ascii=False))
